### This notebook computes "VCA6. Percentage variation rate of the Plant Factor" indicator for the 27 basins of IKI Project

Spanish: 

**Created:** 12/2025 by Sophia Bakar (sbakar@rti.org) 

**Project #:** 0219481  

**Last modified:** 1/12/2026 by Sophia

**Status:** Complete and loaded into the SQL database for the baseline and first future scenario.

**QA Status:** reviewed by  

**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Vulnerabilidad

**Packages:** pandas, numpy, geopandas, sqlite3
 
**Inputs:**   
 
**Assumptions:** 
 
**Future work:** 
 
**Notes:** 

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import sqlite3
import matplotlib.pyplot as plt

In [ ]:
# set up user and database path
#user = 'jmayo'
#user= 'cpickering'
#user = 'sgilson'
#user = 'nreynolds'
user = 'sbakar'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"
# wateralloc_db = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"
wateralloc_db = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"

In [ ]:
# set up indicator ID and get scenarios from database
IndID= 506 #Indicator ID (Exposure = 2 + 0X where X is the Exposure Indicator number, Peligro= 1 +0x, VSB= 3 +0x, VSS= 4 +0x, VCA= 5 +0x)
conn = sqlite3.connect(db_path)

scenarios_df = pd.read_sql_query(
    """
    SELECT ScnID, ScnName
    FROM ScnMod
    """,
    conn
)

conn.close()

# For now: only baseline and first future
#scenario_ids = scenarios_df.loc[
#    scenarios_df['ScnID'].isin([1, 2]), 'ScnID'
#].tolist()

# for all scenarios:
scenario_ids = scenarios_df['ScnID'].tolist()

In [ ]:
## check what scenarios are available in the WaterALLOC database
# Connect to WaterALLOC database
conn_wa = sqlite3.connect(wateralloc_db)

# Query available scenarios
scenarios_query = """
SELECT DISTINCT Scenario
FROM Scenarios
ORDER BY Scenario
"""

wa_scenarios_df = pd.read_sql_query(scenarios_query, conn_wa)

print("Available scenarios in WaterALLOC DB:")
for s in wa_scenarios_df["Scenario"]:
    print(f" - {s}")

# Close WaterALLOC database connection
conn_wa.close()

In [ ]:
# define filepaths for input data


In [ ]:
#INPUTS
subbasins_shapefile = f'C:/Users/{user}/Research Triangle Institute/IKI Peru Project - General/Interno/AI2b_Modelacion/Grupos_Modelacion/GIS_WaterALLOC_General/Peru_AHD_with_districts.shp'
subbasins_gdf = gpd.read_file(subbasins_shapefile).to_crs('EPSG:32718')


#Path to potential output: "C:\Users\sgilson\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Vulnerabilidad\Capitulo-3-Generacion-Electrica-2024.xlsx"

#Path to CH shapefile to map CHs above to actual locations (might need to modify names): https://researchtriangleinstitute.sharepoint.com/:u:/r/sites/IKIPeruProject/Shared%20Documents/General/Interno/AI2b_Modelacion/Datos/COES/Mapa_Minero_Energetico/Shapefiles_COES/COES_CH_EnOperacion.shp?csf=1&web=1&e=bjfG86

#Produccion/WaterALLOC Output: From Enrique on the WaterALLOC output is in BalanceHidrico.sqlite
#The table "WAMSS_Balance por COMID (+Indice de estres)" has information organized by RunID, which can be associated 
# to the scenarios that we are simulating (using WAMMS_RunsInfo).  
# #This table has for each COMID and for each timestep "Oferta Entrada", which is the flow coming into the COMID.  
# Aditionally, it has "Oferta Local Sup", which is flow generated locally and the demand, if the flow leaving the COMID is needed. 


In [ ]:
#Function to remove accents to hopefully aid in mapping CHs in excel to CHs in shapefile: 
#might also want to remove capitalization differences

# Function to remove accents from strings
def remove_accents(text):
    if pd.isna(text):
        return text
    # Normalize to NFD (decomposed form) and filter out combining characters
    return ''.join(c for c in unicodedata.normalize('NFD', str(text)) 
                if unicodedata.category(c) != 'Mn')


SQLITE INSERT 

In [ ]:
#Update relevant dataframe and value column from the calculations above for the specific indicator
insert_data= subbasins_gdf #COMID and Feauture_Count
value_column= 'Feature_Count'

In [ ]:
# Connect to your SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Fixed IndID and ScnID (update above)

#For now I am commenting out the looping of scenarios as I imagine we might run just baseline scenarios for now.
# Define the mapping between scenario and DataFrame column
# scenario_columns = {
#     1: 'ACTUAL ANUAL',
#     2: '2030 ANUAL',
#     3: '2050 ANUAL'
# }

# Prepare list of rows to insert
rows_to_insert = []

#for scn_id, column_name in scenario_columns.items():
for _, row in insert_data.iterrows():
    comid = row['COMID']
    value = row[value_column]
    #normalized_value = None  # or compute something like: value / max_value #took out normalized value from the table for now
    rows_to_insert.append((ScnID, IndID, comid, value))

# Insert data into IndValues_Dyn
insert_query = """
INSERT OR REPLACE INTO IndValues_Dyn (ScnID, IndID, COMID, Value)
VALUES (?, ?, ?, ?);
"""


In [21]:
# Check for duplicates in the input dataframe before insert
df_check = pd.DataFrame(rows_to_insert, columns=['ScnID', 'IndID', 'COMID', 'Value'])
duplicates = df_check.duplicated(subset=['ScnID', 'IndID', 'COMID'])
print("Duplicates in rows_to_insert:", df_check[duplicates])

Duplicates in rows_to_insert: Empty DataFrame
Columns: [ScnID, IndID, COMID, Value]
Index: []


In [ ]:
#Execute insert to SQLite Database
cursor.executemany(insert_query, rows_to_insert)
conn.commit()
conn.close()